# Receipt Reader trên Google Colab

Notebook này tải mã nguồn, cài Tesseract tiếng Việt, build React, chạy Flask + SQLite sau Nginx và tạo một Cloudflare Quick Tunnel. Chạy lần lượt từng ô từ trên xuống.

> Repository đang ở chế độ private nên ô đầu tiên sẽ hỏi GitHub Personal Access Token bằng trường nhập ẩn. Fine-grained token chỉ cần quyền **Contents: Read-only** cho repository `Receipt_reader`. Quick Tunnel là link demo tạm thời, mất hiệu lực khi Colab ngắt kết nối. Không dùng hóa đơn chứa dữ liệu nhạy cảm trong một demo công khai.

In [ ]:
from getpass import getpass
from pathlib import Path
import base64
import os
import subprocess

REPO_URL = "https://github.com/Strangerinhp/Receipt_reader.git"
BRANCH = "main"
REPO_DIR = Path("/content/Receipt_reader")
PRIVATE_REPOSITORY = True  # @param {type:"boolean"}

def clone_repository():
    if REPO_DIR.exists():
        if (REPO_DIR / "backend/requirements.sqlite.txt").is_file():
            print(f"Dùng mã nguồn đã tải tại {REPO_DIR}.")
            return
        if any(REPO_DIR.iterdir()):
            raise RuntimeError("Thư mục đích có dữ liệu chưa hoàn chỉnh. Đổi REPO_DIR hoặc kiểm tra thư mục trước khi thử lại.")

    environment = os.environ.copy()
    environment["GIT_TERMINAL_PROMPT"] = "0"
    token = ""
    encoded = ""
    try:
        if PRIVATE_REPOSITORY:
            token = getpass("GitHub token mới (ẩn): ").strip()
            if not token:
                raise ValueError("Cần token có quyền Contents: Read-only cho Receipt_reader.")
            encoded = base64.b64encode(f"x-access-token:{token}".encode()).decode()
            # Git HTTPS uses Basic auth: username + token as password.
            # Pass the header through the child environment, never command arguments.
            environment["GIT_CONFIG_COUNT"] = "1"
            environment["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraHeader"
            environment["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {encoded}"
        result = subprocess.run(
            ["git", "-c", "credential.helper=", "clone", "--depth", "1",
             "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
            env=environment, capture_output=True, text=True, check=False,
        )
        if result.returncode:
            detail = result.stderr
            for secret in (token, encoded):
                if secret:
                    detail = detail.replace(secret, "[REDACTED]")
            raise RuntimeError(
                f"Git clone thất bại (mã {result.returncode}).\n{detail}\n"
                "Kiểm tra token còn hạn, chọn đúng repository Receipt_reader và quyền Contents: Read-only."
            ) from None
        print(f"Đã tải mã nguồn vào {REPO_DIR}")
    finally:
        environment.clear()
        token = encoded = ""

clone_repository()


In [ ]:
from pathlib import Path
import os
import platform
import stat
import subprocess
import sys
import urllib.request

def run_setup(command, **kwargs):
    # Forward child output through Python so Colab displays the real error.
    print("\\n> " + " ".join(map(str, command)), flush=True)
    with subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1, **kwargs,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        return_code = process.wait()
    if return_code:
        raise RuntimeError(
            f"Bước {command[0]} thất bại (mã {return_code}). Xem log ngay phía trên."
        )

run_setup(["apt-get", "update", "-qq"])
run_setup(
    ["apt-get", "install", "-y", "-qq", "nginx", "tesseract-ocr", "tesseract-ocr-vie"],
)
run_setup(
    [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "-r", str(REPO_DIR / "backend/requirements.sqlite.txt")],
)

machine = platform.machine().lower()
cloudflared_asset = "cloudflared-linux-arm64" if machine in {"aarch64", "arm64"} else "cloudflared-linux-amd64"
cloudflared_path = Path("/content/cloudflared")
urllib.request.urlretrieve(
    f"https://github.com/cloudflare/cloudflared/releases/latest/download/{cloudflared_asset}",
    cloudflared_path,
)
cloudflared_path.chmod(cloudflared_path.stat().st_mode | stat.S_IEXEC)

run_setup(["npm", "ci", "--legacy-peer-deps", "--no-audit", "--no-fund"], cwd=REPO_DIR / "frontend")
build_environment = os.environ.copy()
build_environment.update({"REACT_APP_BACKEND_URL": "/api", "CI": "false"})
run_setup(["npm", "run", "build"], cwd=REPO_DIR / "frontend", env=build_environment)
print("Đã cài dependency và build frontend thành công.")

In [ ]:
from IPython.display import HTML, display
from pathlib import Path
import os
import re
import subprocess
import sys
import time
import urllib.request

def stop_process(name):
    process = globals().get(name)
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()

for process_name in ("cloudflared_process", "nginx_process", "backend_process"):
    stop_process(process_name)
for handle_name in ("cloudflared_log_handle", "nginx_log_handle", "backend_log_handle"):
    handle = globals().get(handle_name)
    if handle is not None and not handle.closed:
        handle.close()

data_dir = Path("/content/receipt_reader_data")
data_dir.mkdir(parents=True, exist_ok=True)
backend_environment = os.environ.copy()
backend_environment.update({
    "DATABASE_ENGINE": "sqlite",
    "SQLITE_DATABASE_PATH": str(data_dir / "invoice_ocr.db"),
    "AUTO_INIT_DB": "true",
    "MAX_UPLOAD_MB": "25",
    "FRONTEND_ORIGIN": "*",
})

backend_log_handle = open("/content/receipt_reader_backend.log", "w")
backend_process = subprocess.Popen(
    [sys.executable, "-m", "gunicorn", "--chdir", str(REPO_DIR / "backend"),
     "--bind", "127.0.0.1:5000", "--workers", "1", "--timeout", "600", "run:app"],
    env=backend_environment, stdout=backend_log_handle, stderr=subprocess.STDOUT,
)

health_url = "http://127.0.0.1:5000/api/health"
for _ in range(60):
    if backend_process.poll() is not None:
        raise RuntimeError(Path("/content/receipt_reader_backend.log").read_text(errors="replace"))
    try:
        with urllib.request.urlopen(health_url, timeout=2) as response:
            if response.status == 200:
                break
    except Exception:
        time.sleep(1)
else:
    raise TimeoutError("Backend không khởi động trong 60 giây.")

nginx_config = Path("/content/receipt_reader_nginx.conf")
nginx_config.write_text(f'''
worker_processes 1;
pid /content/receipt_reader_nginx.pid;
error_log /content/receipt_reader_nginx_error.log;
events {{ worker_connections 1024; }}
http {{
    include /etc/nginx/mime.types;
    access_log /content/receipt_reader_nginx_access.log;
    server {{
        listen 127.0.0.1:7860;
        server_name _;
        root {REPO_DIR / 'frontend/build'};
        index index.html;
        client_max_body_size 25m;
        location /api/ {{
            proxy_pass http://127.0.0.1:5000/api/;
            proxy_http_version 1.1;
            proxy_set_header Host $host;
            proxy_set_header X-Real-IP $remote_addr;
            proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
            proxy_read_timeout 600s;
        }}
        location / {{ try_files $uri $uri/ /index.html; }}
    }}
}}
''', encoding="utf-8")

nginx_log_handle = open("/content/receipt_reader_nginx.log", "w")
nginx_process = subprocess.Popen(
    ["nginx", "-c", str(nginx_config), "-g", "daemon off;"],
    stdout=nginx_log_handle, stderr=subprocess.STDOUT,
)
for _ in range(30):
    try:
        with urllib.request.urlopen("http://127.0.0.1:7860", timeout=2) as response:
            if response.status == 200:
                break
    except Exception:
        time.sleep(1)
else:
    raise TimeoutError("Nginx không khởi động trong 30 giây.")

cloudflare_home = Path("/content/receipt_reader_cloudflare_home")
cloudflare_home.mkdir(parents=True, exist_ok=True)
cloudflare_environment = os.environ.copy()
cloudflare_environment["HOME"] = str(cloudflare_home)
cloudflared_log_path = Path("/content/receipt_reader_cloudflared.log")
cloudflared_log_handle = open(cloudflared_log_path, "w")
cloudflared_process = subprocess.Popen(
    [str(cloudflared_path), "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:7860"],
    env=cloudflare_environment, stdout=cloudflared_log_handle, stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(90):
    cloudflared_log_handle.flush()
    log_text = cloudflared_log_path.read_text(errors="replace")
    match = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", log_text)
    if match:
        public_url = match.group(0)
        break
    if cloudflared_process.poll() is not None:
        raise RuntimeError(log_text)
    time.sleep(1)
if not public_url:
    raise TimeoutError("Cloudflare không trả về URL trong 90 giây. Hãy xem /content/receipt_reader_cloudflared.log")

print("Demo đang chạy. Giữ Colab runtime kết nối để link tiếp tục hoạt động:")
display(HTML(f'<p><a href="{public_url}" target="_blank" style="font-size:20px;font-weight:700">Mở Receipt Reader: {public_url}</a></p>'))

In [ ]:
# Chỉ chạy ô này khi muốn tắt demo.
for process_name in ("cloudflared_process", "nginx_process", "backend_process"):
    stop_process(process_name)
for handle_name in ("cloudflared_log_handle", "nginx_log_handle", "backend_log_handle"):
    handle = globals().get(handle_name)
    if handle is not None and not handle.closed:
        handle.close()
print("Đã tắt Receipt Reader và Cloudflare Tunnel.")